<a href="https://colab.research.google.com/github/TDP-2026-Repository/TECHNOLOGY-DESIGN-PROJECT/blob/main/kevin_few_shot_classification_gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('fpb_test.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,text,label,source
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank


### Install necessary libraries

We need to install the `transformers` library to work with the Gemma model and `accelerate` for optimized model loading and inference.

In [3]:
# install transformers
!pip install transformers accelerate -q

### Load the Gemma model and tokenizer

Now, let's load the `google/gemma-3-4b-it` model and its corresponding tokenizer. This model is designed for instruction following, which is suitable for few-shot classification tasks.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# model id
model_name = "google/gemma-3-4b-it"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

### data prep, few-shot

create a prompt that includes a few examples of the task (the 'shots') and then the text we want the model to classify

split the dataset into a small set for examples and the rest for classification

In [5]:
# unique labels are...
print("Unique labels in the dataset:", df['label'].unique())
unique_labels = df['label'].unique().tolist()

# Shuffle the dataframe to ensure random selection of few-shot examples
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Select a few examples for few-shot prompting
num_few_shot_examples = 10 # change this number based on.... uh......... insight tm
few_shot_examples = df_shuffled.head(num_few_shot_examples)
inference_data = df_shuffled.tail(len(df_shuffled) - num_few_shot_examples).reset_index(drop=True)

# Define the prompt prefix using the few-shot examples
few_shot_prompt_prefix = f"Classify the following text into one of these categories: {', '.join(unique_labels)}.\nThe first few samples have been pre-filled for you.\n\n"

for index, row in few_shot_examples.iterrows():
    few_shot_prompt_prefix += f"Text: '{row['text']}'\nLabel: {row['label']}\n\n"

print(f"\nPrompt:{few_shot_prompt_prefix}")
print(f"Number of inference samples: {len(inference_data)}")

Unique labels in the dataset: ['Neutral' 'Fear' 'Optimism' 'Sadness' 'Joy']

Prompt:Classify the following text into one of these categories: Neutral, Fear, Optimism, Sadness, Joy.
The first few samples have been pre-filled for you.

Text: 'finnish kci konecranes has been awarded an order for four hot metal ladle cranes by indian steel producer bhushan steel strips to be delivered in 2007 .'
Label: Optimism

Text: 'the acquisition is expected to take place by the end of august 2007 .'
Label: Neutral

Text: 'in the automobile space , maruti is the most searched brand in cars .'
Label: Neutral

Text: 'kesko agro eesti , the retailer and wholesaler of grain , agricultural and warehousing machinery and accessories , had net sales of 81 million euros in 2007 , an increase by one-tenth over the preceding year .'
Label: Optimism

Text: 'the daily graphic newspaper , in october , reported an initiative being embarked upon by the fidelity bank to partner ghana post , which has offices across th

### Performing Few-Shot Inference

Now we will iterate through our `inference_data`, generate a prompt for each text using our few-shot examples, and get a prediction from the Gemma model. The model will complete the prompt by predicting the 'Label:' part.

In [6]:
!pip install -q transformers accelerate bitsandbytes scikit-learn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# classify function
def classify(text, candidate_labels):
    prompt = f"Classify the following text into one of these categories: {', '.join(candidate_labels)}.\n\nText: {text}\nCategory:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    prediction = response.split("Category:")[-1].strip().lower()

    # Find the best match in candidate labels
    for label in candidate_labels:
        if label.lower() in prediction:
            return label
    return "unknown" # did something go wrong?

# time to evaluate!
y_true = inference_data['label'].tolist() # all the correct labels
y_pred = [classify(text, unique_labels) for text in inference_data['text']] # all the predicted labels

# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

# export metrics
results_dict = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [accuracy, precision, recall, f1]
}

# create dataframe
results_df = pd.DataFrame(results_dict)

# export to csv
results_df.to_csv('few_shot_gemma_01.csv', index=False)

print("Results saved to 'few_shot_gemma_01.csv'")
display(results_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Results saved to 'few_shot_gemma_01.csv'


,Metric,Value
0,Accuracy,0.598326
1,Precision,0.706956
2,Recall,0.598326
3,F1 Score,0.478368
